# 07 · Schema-Stress Suite at Every Precision

**Project:** Silent Tool-Call Failures: How Quantization Degrades Function Calling in Small LLMs
**Goal:** run the schema-stress suite (175 questions in 7 groups, `data/stress_suite/`) on all 25 model × precision combinations, in free and in constrained decoding (Update 12), to see which schema features break first as precision drops.

Everything else is the same as the main runs: prompt, greedy decoding, scorer, model revisions and quantization settings. Run ids are `<config>_<mode>_stress`.

| Precision | Where the weights come from |
|---|---|
| FP16, NF4 | the pinned Hugging Face model (NF4 is quantized by vLLM while loading) |
| INT8 | made here by `src/quantize.py` (under a minute per model) |
| GPTQ | downloaded from the private Hugging Face repositories made in Update 9, at a pinned commit |
| AWQ | downloaded from the private Hugging Face repositories made in Update 6, at a pinned commit |

**Check first:** the first model answers 2 questions per group in both modes. If that fails, the notebook stops before the long runs. (All 175 schemas were compiled with XGrammar 0.1.18 on a PC before this notebook was written.)

## How to run on Kaggle

The 50 runs are split into two parts, as in notebook 06, so that one Kaggle session cannot lose all of them.

1. kaggle.com → **Create → New Notebook → File → Import Notebook**, then upload this file. Do this twice.
2. In the right panel, under **Settings**: **Accelerator → GPU T4 x2** and **Internet → On**.
3. **Add-ons → Secrets:** switch on `HF_TOKEN` for each notebook (needed for Llama and for the private checkpoints).
4. In the settings cell of the second notebook, set `PART = "B"`.
5. Click **Save Version → Save & Run All (Commit)** in both.

**Time:** not measured yet. The suite has 175 questions, an eighth of the main set, but each model is loaded 10 times (5 precisions × 2 modes), and in constrained mode XGrammar first compiles one grammar per question: on a PC CPU a 50-function menu takes about 15 seconds, so each constrained run spends several minutes before it starts answering. A rough guess is 3 to 5 hours per part, plus about 20 minutes of installation.
**Output:** `update12_results_A.zip` and `update12_results_B.zip`, with the raw answers, scores, logs and `stress_A.md` / `stress_B.md`.

## 1. Settings
Edit this cell only.

In [1]:
import os

REPO_URL = "https://github.com/shawalkabirchy/quantization-functional-call"
REPO_REF = "main"             # branch, tag or commit of the code to run

# Two runs, one per part: "A" and "B". The parts can run at the same time, in two notebooks.
PART = "B"
MODELS_OF_PART = {"A": ["qwen3-1.7b", "llama-3.2-1b", "llama-3.2-3b"], "B": ["qwen3-4b", "qwen3-8b"]}
# Grouped by model, so each model is downloaded once
CONFIGS = [f"configs/{m}_{p}.yaml" for m in MODELS_OF_PART[PART] for p in ["fp16", "int8", "nf4", "gptq", "awq"]]
MODES = ["free", "constrained"]
CHECK_LIMIT = 2               # questions per group in the quick check
REPORT_NAME = f"stress_{PART}.md"
RUN_TIMEOUT_MIN = 90          # a run that takes longer than this is stopped, and the next one starts
QUANTIZE_TIMEOUT_MIN = 30     # the same for making an INT8 checkpoint
DELETE_MODEL_AFTER_RUN = True # frees disk space: deletes each checkpoint after its runs, each model after its last run

# 4-bit checkpoints made in Updates 6 (AWQ) and 9 (GPTQ), private on Hugging Face, pinned to the commit used there
HF_CHECKPOINT_USER = "shawalkabir"
CHECKPOINT_COMMITS = {
    "qwen3-1.7b_gptq": "7803089b341f86527b23c2a49f9bd4cc95662352",
    "qwen3-4b_gptq": "a82f18018d851518178ef4ce6cae34118c55da21",
    "qwen3-8b_gptq": "bec7853f0aae7bd4b51c30b6301f386caf584f53",
    "llama-3.2-1b_gptq": "b14742c0f416130b2e2ce2c595be390f15463c2b",
    "llama-3.2-3b_gptq": "59872d087ea389a4a30197553d7fe1b6d82770e7",
    "qwen3-1.7b_awq": "3cc45f1ddb6c1fcd967a26f5b649556d884360f2",
    "qwen3-4b_awq": "e8a7317cb3f6f927bea3e442b00e487d4a2d9ec1",
    "qwen3-8b_awq": "6f1d2a83ad23b41709e98366253f7f86fcf0a9ea",
    "llama-3.2-1b_awq": "e8cd172864e551542651600ebfbcbfc667328c90",
    "llama-3.2-3b_awq": "6f9fd9e999f07abdfe75085e85e0f9f784cad923",
}

BFCL_VERSION = "2026.3.23"    # pinned for reproducibility
BITSANDBYTES_VERSION = "0.45.5"  # same vLLM environment as notebooks 04-06
ACCELERATE_VERSION = "1.6.0"
WORK = "/kaggle/working"
REPO = f"{WORK}/repo"                         # code and results (kept as notebook output)
VENV = os.path.expanduser("~/bfcl-venv")      # vLLM + BFCL environment (not kept)
CHECKPOINT_DIR = os.path.expanduser("~/checkpoints")   # INT8, GPTQ and AWQ checkpoints (not kept: they are large)
os.environ["CHECKPOINT_DIR"] = CHECKPOINT_DIR           # read by src/quantize.py and src/run_eval.py


def run_id_of(config, mode):
    return os.path.splitext(os.path.basename(config))[0] + f"_{mode}_stress"

## 2. Check the GPU, internet and the Hugging Face token

In [2]:
import json
import shutil
import subprocess
import time
from datetime import date

import requests

smi = subprocess.run(
    ["nvidia-smi", "--query-gpu=index,name,memory.total,driver_version", "--format=csv,noheader"],
    capture_output=True, text=True,
)
if smi.returncode != 0:
    raise RuntimeError("No GPU found. Set Settings -> Accelerator -> GPU T4 x2, then run again.")
print("GPUs:\n" + smi.stdout)

try:
    requests.get("https://pypi.org/simple/", timeout=15).raise_for_status()
    print("Internet: OK")
except Exception as e:
    raise RuntimeError("No internet. Set Settings -> Internet -> On (needs a phone-verified Kaggle account).") from e

try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN: loaded from Kaggle Secrets")
except Exception:
    raise RuntimeError("HF_TOKEN is not set. Llama 3.2 and the private checkpoints need it (Add-ons -> Secrets).")

GPUs:
0, Tesla T4, 15360 MiB, 580.178.04
1, Tesla T4, 15360 MiB, 580.178.04

Internet: OK
HF_TOKEN: loaded from Kaggle Secrets


## 3. Get the code and check access to every model
Clones the repository, then downloads one small file of each model at the pinned revision. This finds a missing token or missing Llama approval now, not after an hour of running.

In [3]:
import yaml
from huggingface_hub import hf_hub_download

shutil.rmtree(REPO, ignore_errors=True)
!git clone --quiet {REPO_URL} {REPO}
!git -C {REPO} checkout --quiet {REPO_REF}
!git -C {REPO} log -1 --format="Code version: %h %s (%ci)"

cfgs = {}
problems = []
for config in CONFIGS:
    with open(f"{REPO}/{config}") as f:
        cfgs[config] = yaml.safe_load(f)
    model, revision = cfgs[config]["model"], str(cfgs[config]["revision"])
    try:
        hf_hub_download(model, "config.json", revision=revision, token=os.environ.get("HF_TOKEN"))
        print(f"  access OK  {model}")
    except Exception as e:
        problems.append(f"{model}: {type(e).__name__}: {str(e)[:200]}")
if problems:
    print("\n".join(problems))
    raise RuntimeError(
        "Some models cannot be downloaded. For Llama: add HF_TOKEN under Add-ons -> Secrets, switch it on for "
        "this notebook, and check that https://huggingface.co/meta-llama/Llama-3.2-1B-Instruct says you have access."
    )

# The stress suite comes with the code: 7 groups of 25 questions
for group in ["stress_plain", "stress_nested", "stress_enum", "stress_number", "stress_optional", "stress_unicode",
              "stress_menu"]:
    with open(f"{REPO}/data/stress_suite/{group}.json", encoding="utf-8") as f:
        n = sum(1 for _ in f)
    assert n == 25, (group, n)
print("Stress suite: 7 groups of 25 questions")

Code version: bae21f3 Update 11: schema-stress suite v1 (175 questions in 7 groups); Introduction drafted (2026-11-06 20:03:00 +0600)


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

  access OK  Qwen/Qwen3-4B
  access OK  Qwen/Qwen3-4B
  access OK  Qwen/Qwen3-4B
  access OK  Qwen/Qwen3-4B
  access OK  Qwen/Qwen3-4B


config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

  access OK  Qwen/Qwen3-8B
  access OK  Qwen/Qwen3-8B
  access OK  Qwen/Qwen3-8B
  access OK  Qwen/Qwen3-8B
  access OK  Qwen/Qwen3-8B
Stress suite: 7 groups of 25 questions


## 4. Install the vLLM environment (about 1-5 minutes)
Exactly the same environment as notebooks 04-06: a separate Python 3.11 environment with `bfcl-eval[oss_eval_vllm]` (vLLM 0.8.5) and the same pins. No GPTQModel environment is needed here: the GPTQ and AWQ checkpoints are downloaded, and INT8 is made with bitsandbytes in this environment.

In [4]:
!pip install -q uv
shutil.rmtree(VENV, ignore_errors=True)
!uv venv {VENV} --python 3.11
!uv pip install --python {VENV}/bin/python "bfcl-eval[oss_eval_vllm]=={BFCL_VERSION}" "transformers==4.51.3" "openai>=1.86.0,<2" "fastapi<0.116" "pydantic<2.12" "setuptools>=77.0.3,<80" soundfile "bitsandbytes=={BITSANDBYTES_VERSION}" "accelerate=={ACCELERATE_VERSION}"

CHECK = (
    "import vllm, bfcl_eval, yaml, xgrammar, triton, torch, bitsandbytes, accelerate; "
    "from importlib.metadata import version; "
    "print('vllm', version('vllm'), '| xgrammar', version('xgrammar'), '| torch', version('torch'), "
    "'| bitsandbytes', version('bitsandbytes'), '| accelerate', version('accelerate')); "
    "print('GPUs:', [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])"
)
r = subprocess.run([f"{VENV}/bin/python", "-c", CHECK], capture_output=True, text=True)
print(r.stdout)
if r.returncode != 0:
    print(r.stderr[-5000:])
    raise RuntimeError("Environment check failed (see above). Check the output of the install step.")

freeze = subprocess.run(["uv", "pip", "freeze", "--python", f"{VENV}/bin/python"], capture_output=True, text=True)
with open(f"{WORK}/requirements-freeze.txt", "w") as f:
    f.write(freeze.stdout)
print(f"Saved {len(freeze.stdout.splitlines())} package versions to {WORK}/requirements-freeze.txt")

Using CPython 3.11.18
Creating virtual environment at: /root/bfcl-venv
Activate with: source /root/bfcl-venv/bin/activate
Using Python 3.11.18 environment at: /root/bfcl-venv
Resolved 215 packages in 2.36s
Prepared 215 packages in 5m 14s
Installed 215 packages in 538ms
 + accelerate==1.6.0
 + agent-detector==2.0.0
 + aiohappyeyeballs==2.7.1
 + aiohttp==3.14.5
 + aiosignal==1.4.0
 + airportsdata==20260905
 + annotated-doc==0.0.5
 + annotated-types==0.8.0
 + anthropic==2.6.0
 + anyio==4.16.0
 + argcomplete==3.7.2
 + astor==0.8.1
 + attrs==26.1.0
 + beautifulsoup4==4.15.0
 + bfcl-eval==2026.3.23
 + bitsandbytes==0.45.5
 + black==26.10.1
 + blake3==1.0.11
 + boto3==1.43.126
 + botocore==1.43.126
 + cachetools==8.0.0
 + certifi==2026.10.23
 + cffi==2.1.1
 + charset-normalizer==3.5.2
 + click==8.5.1
 + cloudpickle==3.1.2
 + cohere==5.18.0
 + compressed-tensors==0.9.3
 + cryptography==50.0.2
 + cupy-cuda12x==13.6.0
 + dashscope==2.0.4
 + datamodel-code-generator==0.25.7
 + deprecated==1.3.1
 

## 5. Quick check before the long runs
The first config answers 2 questions per group in both modes; the answers go to `check/`, apart from the real results. If the check fails, the notebook stops here.

In [5]:
def tail(path, n=25):
    with open(path, errors="replace") as f:
        return "".join(f.readlines()[-n:])


def run_logged(args, log_path, timeout_min, venv=VENV):
    """Runs a script of the repository in an environment; returns its exit code, or None after a timeout."""
    with open(log_path, "w") as log:
        try:
            return subprocess.run([f"{venv}/bin/python", *args], cwd=REPO, stdout=log, stderr=subprocess.STDOUT,
                                  timeout=timeout_min * 60).returncode
        except subprocess.TimeoutExpired:
            print(f"  stopped after {timeout_min} minutes")
            return None


def gpu_memory_used_mib():
    out = subprocess.run(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
                         capture_output=True, text=True).stdout
    return [int(x) for x in out.split()]


def wait_for_free_gpus(limit_mib=1000, wait_s=120):
    """A finished run should leave the GPUs empty; wait briefly if they are not yet."""
    start = time.time()
    while max(gpu_memory_used_mib()) > limit_mib and time.time() - start < wait_s:
        time.sleep(5)
    used = gpu_memory_used_mib()
    if max(used) > limit_mib:
        print(f"  WARNING: GPU memory still in use before this run: {used} MiB")


def make_checkpoint(config, stem, log_path):
    """Downloads a GPTQ or AWQ checkpoint at its pinned commit, or makes the INT8 one. Returns True if it worked."""
    method = cfgs[config]["quantize"]["method"]
    if method in ("gptq", "awq"):
        from huggingface_hub import snapshot_download
        repo_id = f"{HF_CHECKPOINT_USER}/{cfgs[config]['model'].split('/')[-1]}-{method.upper()}-W4A16-g128"
        with open(log_path, "w") as log:
            try:
                if not os.path.exists(f"{CHECKPOINT_DIR}/{stem}/quantize_info.json"):
                    snapshot_download(repo_id, revision=CHECKPOINT_COMMITS[stem], local_dir=f"{CHECKPOINT_DIR}/{stem}",
                                      token=os.environ.get("HF_TOKEN"))
                log.write(f"Using {repo_id} at {CHECKPOINT_COMMITS[stem][:7]}\n")
                return True
            except Exception as e:
                log.write(f"Could not download {repo_id}: {type(e).__name__}: {str(e)[:300]}\n")
                return False
    return run_logged(["src/quantize.py", "--config", config], log_path, QUANTIZE_TIMEOUT_MIN) == 0


# The first config answers 2 questions per group in both modes
CHECK_DIR = f"{WORK}/check"
shutil.rmtree(CHECK_DIR, ignore_errors=True)
config, stem = CONFIGS[0], os.path.splitext(os.path.basename(CONFIGS[0]))[0]
for mode in MODES:
    print(f"\n=== check {stem}, {mode}: started {time.strftime('%H:%M:%S')} UTC", flush=True)
    wait_for_free_gpus()
    log_path = f"{WORK}/check_{stem}_{mode}.log"
    ok = run_logged(["src/run_eval.py", "--config", config, "--mode", mode, "--categories", "stress",
                     "--limit", str(CHECK_LIMIT), "--results-dir", CHECK_DIR, "--overwrite"], log_path, 30) == 0
    if not ok:
        print(f"End of {log_path}:\n{tail(log_path, 40)}")
        raise RuntimeError("The quick check failed; stopping before the long runs.")
    run_id = f"{run_id_of(config, mode)}_limit{CHECK_LIMIT}"
    r = subprocess.run([f"{VENV}/bin/python", "src/score.py", f"{CHECK_DIR}/raw/{run_id}.jsonl"],
                       cwd=REPO, capture_output=True, text=True)
    print(r.stdout[-1500:])
    if r.returncode != 0:
        print(r.stderr[-3000:])
        raise RuntimeError("Scoring the quick check failed; stopping before the long runs.")
    with open(f"{CHECK_DIR}/raw/{run_id}.jsonl") as f:
        for line in list(f)[:2]:
            row = json.loads(line)
            print(f"  {row['id']}: {row['output'][:200]!r}")
print("\nQuick check passed.")


=== check qwen3-4b_fp16, free: started 17:11:37 UTC

qwen3-4b_fp16_free_stress_limit2  (14 questions)
  category               correct  accuracy   parsed
  stress_plain               2/2    100.00   100.00
  stress_nested              2/2    100.00   100.00
  stress_enum                2/2    100.00   100.00
  stress_number              2/2    100.00   100.00
  stress_optional            2/2    100.00   100.00
  stress_unicode             2/2    100.00   100.00
  stress_menu                2/2    100.00   100.00
  saved /kaggle/working/check/scores/qwen3-4b_fp16_free_stress_limit2.jsonl and /kaggle/working/check/summary.csv

  stress_plain_0: '[{"name": "book_table", "arguments": {"restaurant": "Saffron Lane", "party_size": 8, "time": "20:00"}}]'
  stress_plain_1: '[{"name": "get_weather_forecast", "arguments": {"city": "Seoul", "days": 10}}]'

=== check qwen3-4b_fp16, constrained: started 17:13:29 UTC

qwen3-4b_fp16_constrained_stress_limit2  (14 questions)
  category               c

## 6. Run and score every config in both modes (a few hours)
One config at a time: the checkpoint (not needed for FP16 and NF4), then for each mode `run_eval.py --categories stress` (answers) and `score.py` (accuracy per group). Every step's full log is saved. If a run fails, the notebook prints the end of its log and goes on with the next one.

In [6]:
status, minutes = {}, {}
last_config_of = {cfgs[c]["model"]: c for c in CONFIGS}   # each model is deleted after its last config
for config in CONFIGS:
    cfg = cfgs[config]
    stem = os.path.splitext(os.path.basename(config))[0]
    ok = True
    if "quantize" in cfg:
        print(f"\n=== {stem}: checkpoint, started {time.strftime('%H:%M:%S')} UTC", flush=True)
        wait_for_free_gpus()
        log_path = f"{WORK}/quantize_{stem}.log"
        ok = make_checkpoint(config, stem, log_path)
        print("  " + tail(log_path, 1).strip() if ok else f"  checkpoint FAILED. End of {log_path}:\n{tail(log_path)}")
    for mode in MODES:
        run_id = run_id_of(config, mode)
        if not ok:
            status[run_id] = "CHECKPOINT FAILED"
            continue
        print(f"\n=== {run_id} ({cfg['model']}): started {time.strftime('%H:%M:%S')} UTC", flush=True)
        start = time.time()
        wait_for_free_gpus()
        log_path = f"{WORK}/run_{run_id}.log"
        run_logged(["src/run_eval.py", "--config", config, "--mode", mode, "--categories", "stress", "--overwrite"],
                   log_path, RUN_TIMEOUT_MIN)
        if not os.path.exists(f"{REPO}/results/meta/{run_id}.json"):
            status[run_id] = "FAILED"
            print(f"  FAILED. End of {log_path}:\n{tail(log_path)}")
        else:
            print("  answers: " + next((l.strip() for l in tail(log_path, 10).splitlines() if l.startswith("Done:")), ""))
            r = subprocess.run([f"{VENV}/bin/python", "src/score.py", f"results/raw/{run_id}.jsonl"],
                               cwd=REPO, capture_output=True, text=True)
            print(r.stdout[-1500:])
            status[run_id] = "OK" if r.returncode == 0 else "SCORING FAILED"
            if r.returncode != 0:
                print(r.stderr[-3000:])
        minutes[run_id] = (time.time() - start) / 60
        print(f"  {status[run_id]} after {minutes[run_id]:.1f} min")

    if DELETE_MODEL_AFTER_RUN:
        shutil.rmtree(f"{CHECKPOINT_DIR}/{stem}", ignore_errors=True)
        if last_config_of[cfg["model"]] == config:
            shutil.rmtree(os.path.expanduser(f"~/.cache/huggingface/hub/models--{cfg['model'].replace('/', '--')}"),
                          ignore_errors=True)

print("\nSummary:")
for run_id, s in status.items():
    took = f"  ({minutes[run_id]:.1f} min)" if run_id in minutes else ""
    print(f"  {run_id:36s} {s}{took}")
print(f"  total: {sum(minutes.values()) / 60:.1f} hours of runs")
if any(s != "OK" for s in status.values()):
    print("Some runs did not finish. The results of the others are still reported and packaged below.")


=== qwen3-4b_fp16_free_stress (Qwen/Qwen3-4B): started 17:14:59 UTC
  answers: Done: 175 answers in 91 s (67.5 output tokens/s); finish reasons {'stop': 175}

qwen3-4b_fp16_free_stress  (175 questions)
  category               correct  accuracy   parsed
  stress_plain             25/25    100.00   100.00
  stress_nested            25/25    100.00   100.00
  stress_enum              25/25    100.00   100.00
  stress_number            25/25    100.00   100.00
  stress_optional          25/25    100.00   100.00
  stress_unicode           25/25    100.00   100.00
  stress_menu              25/25    100.00   100.00
  saved /kaggle/working/repo/results/scores/qwen3-4b_fp16_free_stress.jsonl and /kaggle/working/repo/results/summary.csv

  OK after 2.6 min

=== qwen3-4b_fp16_constrained_stress (Qwen/Qwen3-4B): started 17:17:36 UTC
  answers: Done: 175 answers in 215 s (28.5 output tokens/s); finish reasons {'stop': 175}

qwen3-4b_fp16_constrained_stress  (175 questions)
  category            

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 15 files:   0%|          | 0/15 [00:00<?, ?it/s]

  Using shawalkabir/Qwen3-4B-GPTQ-W4A16-g128 at a82f180

=== qwen3-4b_gptq_free_stress (Qwen/Qwen3-4B): started 17:56:06 UTC
  answers: Done: 175 answers in 86 s (71.3 output tokens/s); finish reasons {'stop': 175}

qwen3-4b_gptq_free_stress  (175 questions)
  category               correct  accuracy   parsed
  stress_plain             25/25    100.00   100.00
  stress_nested            19/25     76.00    76.00
  stress_enum              25/25    100.00   100.00
  stress_number            25/25    100.00   100.00
  stress_optional          25/25    100.00   100.00
  stress_unicode           25/25    100.00   100.00
  stress_menu              24/25     96.00    96.00
  saved /kaggle/working/repo/results/scores/qwen3-4b_gptq_free_stress.jsonl and /kaggle/working/repo/results/summary.csv

  OK after 2.5 min

=== qwen3-4b_gptq_constrained_stress (Qwen/Qwen3-4B): started 17:58:35 UTC
  answers: Done: 175 answers in 223 s (27.5 output tokens/s); finish reasons {'stop': 175}

qwen3-4b_gptq_co

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 14 files:   0%|          | 0/14 [00:00<?, ?it/s]

... (3 outputs left out)

  Using shawalkabir/Qwen3-8B-GPTQ-W4A16-g128 at bec7853

=== qwen3-8b_gptq_free_stress (Qwen/Qwen3-8B): started 19:22:05 UTC
  answers: Done: 175 answers in 157 s (38.8 output tokens/s); finish reasons {'stop': 175}

qwen3-8b_gptq_free_stress  (175 questions)
  category               correct  accuracy   parsed
  stress_plain             25/25    100.00   100.00
  stress_nested            25/25    100.00   100.00
  stress_enum              25/25    100.00   100.00
  stress_number            25/25    100.00   100.00
  stress_optional          25/25    100.00   100.00
  stress_unicode           25/25    100.00   100.00
  stress_menu              25/25    100.00   100.00
  saved /kaggle/working/repo/results/scores/qwen3-8b_gptq_free_stress.jsonl and /kaggle/working/repo/results/summary.csv

  OK after 4.2 min

=== qwen3-8b_gptq_constrained_stress (Qwen/Qwen3-8B): started 19:26:16 UTC
  answers: Done: 175 answers in 255 s (24.0 output tokens/s); finish reasons {'stop': 175}

qwen3-8b_gptq_c

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 16 files:   0%|          | 0/16 [00:00<?, ?it/s]

  Using shawalkabir/Qwen3-8B-AWQ-W4A16-g128 at 6f1d2a8

=== qwen3-8b_awq_free_stress (Qwen/Qwen3-8B): started 19:32:15 UTC
  answers: Done: 175 answers in 177 s (34.6 output tokens/s); finish reasons {'stop': 175}

qwen3-8b_awq_free_stress  (175 questions)
  category               correct  accuracy   parsed
  stress_plain             25/25    100.00   100.00
  stress_nested            25/25    100.00   100.00
  stress_enum              25/25    100.00   100.00
  stress_number            25/25    100.00   100.00
  stress_optional          25/25    100.00   100.00
  stress_unicode           25/25    100.00   100.00
  stress_menu              25/25    100.00   100.00
  saved /kaggle/working/repo/results/scores/qwen3-8b_awq_free_stress.jsonl and /kaggle/working/repo/results/summary.csv

  OK after 4.4 min

=== qwen3-8b_awq_constrained_stress (Qwen/Qwen3-8B): started 19:36:38 UTC
  answers: Done: 175 answers in 264 s (23.2 output tokens/s); finish reasons {'stop': 175}

qwen3-8b_awq_constra

## 7. Tables
Accuracy per group for every run, and format, length and speed.

In [7]:
import pandas as pd

GROUPS = {"stress_plain": "Plain", "stress_nested": "Nested", "stress_enum": "Enum", "stress_number": "Number",
          "stress_optional": "Optional", "stress_unicode": "Unicode", "stress_menu": "Menu"}
PRECISIONS = ["fp16", "int8", "nf4", "gptq", "awq"]
summary = pd.read_csv(f"{REPO}/results/summary.csv")
acc_rows, info_rows = [], []
for config in CONFIGS:
    base = os.path.splitext(os.path.basename(config))[0]
    for mode in MODES:
        run_id = run_id_of(config, mode)
        s = summary[summary["run_id"] == run_id].set_index("category")
        if s.empty:
            continue
        with open(f"{REPO}/results/meta/{run_id}.json") as f:
            meta = json.load(f)
        name, label = meta["model"].split("/")[-1], meta["precision"].upper()
        acc_rows.append({"Model": name, "Precision": label, "Mode": mode,
                         **{col: float(s.loc[g, "accuracy"]) for g, col in GROUPS.items() if g in s.index}})
        info_rows.append({"Model": name, "Precision": label, "Mode": mode,
                          "Parsed (%)": round(float((s["parsed_pct"] * s["total"]).sum() / s["total"].sum()), 2),
                          "Cut off at max_tokens": meta["finish_reasons"].get("length", 0),
                          "Weights per GPU (GiB)": meta["memory"]["model_weights_gib"],
                          "Generation (s)": meta["generation_seconds"],
                          "Output tokens/s": meta["output_tokens_per_second"]})
accuracy, info = pd.DataFrame(acc_rows), pd.DataFrame(info_rows)
for mode in MODES:
    print(f"\nAccuracy (%) per group, {mode} decoding")
    display(accuracy[accuracy["Mode"] == mode].drop(columns="Mode"))
display(info)
accuracy.to_csv(f"{WORK}/stress_accuracy_{PART}.csv", index=False)
info.to_csv(f"{WORK}/stress_run_info_{PART}.csv", index=False)


Accuracy (%) per group, free decoding


,Model,Precision,Plain,Nested,Enum,Number,Optional,Unicode,Menu
0,Qwen3-4B,FP16,100.0,100.0,100.0,100.0,100.0,100.0,100.0
2,Qwen3-4B,INT8,100.0,96.0,96.0,100.0,100.0,100.0,100.0
4,Qwen3-4B,NF4,100.0,76.0,100.0,100.0,100.0,96.0,96.0
6,Qwen3-4B,GPTQ,100.0,76.0,100.0,100.0,100.0,100.0,96.0
8,Qwen3-4B,AWQ,100.0,76.0,96.0,100.0,100.0,96.0,100.0
10,Qwen3-8B,FP16,100.0,100.0,100.0,100.0,100.0,100.0,100.0
12,Qwen3-8B,INT8,100.0,100.0,100.0,100.0,100.0,100.0,100.0
14,Qwen3-8B,NF4,100.0,100.0,96.0,100.0,100.0,100.0,100.0
16,Qwen3-8B,GPTQ,100.0,100.0,100.0,100.0,100.0,100.0,100.0
18,Qwen3-8B,AWQ,100.0,100.0,100.0,100.0,100.0,100.0,100.0



Accuracy (%) per group, constrained decoding


,Model,Precision,Plain,Nested,Enum,Number,Optional,Unicode,Menu
1,Qwen3-4B,FP16,100.0,100.0,100.0,100.0,100.0,100.0,100.0
3,Qwen3-4B,INT8,100.0,100.0,96.0,100.0,100.0,100.0,100.0
5,Qwen3-4B,NF4,100.0,100.0,100.0,100.0,100.0,96.0,100.0
7,Qwen3-4B,GPTQ,100.0,100.0,100.0,100.0,100.0,100.0,100.0
9,Qwen3-4B,AWQ,100.0,100.0,96.0,100.0,100.0,96.0,100.0
11,Qwen3-8B,FP16,100.0,100.0,100.0,100.0,100.0,100.0,100.0
13,Qwen3-8B,INT8,100.0,100.0,100.0,100.0,100.0,100.0,100.0
15,Qwen3-8B,NF4,100.0,100.0,96.0,100.0,100.0,100.0,100.0
17,Qwen3-8B,GPTQ,100.0,100.0,100.0,100.0,100.0,100.0,100.0
19,Qwen3-8B,AWQ,100.0,100.0,100.0,100.0,100.0,100.0,100.0


,Model,Precision,Mode,Parsed (%),Cut off at max_tokens,Weights per GPU (GiB),Generation (s),Output tokens/s
0,Qwen3-4B,FP16,free,100.00,0,7.5552,90.6,67.5
1,Qwen3-4B,FP16,constrained,100.00,0,7.5552,214.9,28.5
2,Qwen3-4B,INT8,free,99.43,0,4.1792,133.3,45.9
3,Qwen3-4B,INT8,constrained,100.00,0,4.1792,319.4,19.1
4,Qwen3-4B,NF4,free,96.00,0,2.4923,564.0,10.9
5,Qwen3-4B,NF4,constrained,100.00,0,2.4923,626.8,9.8
6,Qwen3-4B,GPTQ,free,96.00,0,2.5045,85.7,71.3
7,Qwen3-4B,GPTQ,constrained,100.00,0,2.5045,222.6,27.5
8,Qwen3-4B,AWQ,free,96.57,0,2.5045,100.0,61.9
9,Qwen3-4B,AWQ,constrained,100.00,0,2.5045,230.0,26.9


## 8. Write the report and package the results

In [8]:
import glob


def md_table(df):
    def fmt(v):
        if v is None or (isinstance(v, float) and v != v):
            return ""
        return f"{v:.2f}" if isinstance(v, float) else str(v)
    head = "| " + " | ".join(df.columns) + " |"
    sep = "|" + "|".join("---" if i == 0 else "---:" for i in range(len(df.columns))) + "|"
    return [head, sep] + ["| " + " | ".join(fmt(v) for v in row) + " |" for row in df.itertuples(index=False)]


commit = subprocess.run(["git", "-C", REPO, "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
lines = [f"# Schema-stress suite, part {PART} (Update 12)", "", f"**Date:** {date.today()}",
         f"**Code:** commit `{commit}`; 175 questions in 7 groups of 25 (`data/stress_suite/`), free and constrained "
         "decoding, the same models, settings and scorer as the main runs.", ""]
for mode in MODES:
    part = accuracy[accuracy["Mode"] == mode].drop(columns="Mode")
    lines += [f"## Accuracy per group (%), {mode} decoding", ""] + md_table(part) + [""]
lines += ["## Format, length and speed", ""] + md_table(info) + [""]
failed = [r for r, s in status.items() if s != "OK"]
lines += ["## Notes", "", f"- Did not finish: {', '.join(failed)}" if failed else "- All runs finished.", ""]
report = "\n".join(lines) + "\n"
with open(f"{WORK}/{REPORT_NAME}", "w") as f:
    f.write(report)
print(report)

pack = f"{WORK}/update12_results_{PART}"
shutil.rmtree(pack, ignore_errors=True)
shutil.copytree(f"{REPO}/results", f"{pack}/results", ignore=shutil.ignore_patterns("bfcl_repro", "figures"))
if os.path.exists(CHECK_DIR):
    shutil.copytree(CHECK_DIR, f"{pack}/check")
for name in [REPORT_NAME, f"stress_accuracy_{PART}.csv", f"stress_run_info_{PART}.csv", "requirements-freeze.txt"]:
    if os.path.exists(f"{WORK}/{name}"):
        shutil.copy(f"{WORK}/{name}", pack)
for log_path in sorted(glob.glob(f"{WORK}/*.log")):
    shutil.copy(log_path, pack)
archive = shutil.make_archive(pack, "zip", pack)
print("Saved", archive)

# Schema-stress suite, part B (Update 12)

**Date:** 2026-11-07
**Code:** commit `bae21f3`; 175 questions in 7 groups of 25 (`data/stress_suite/`), free and constrained decoding, the same models, settings and scorer as the main runs.

## Accuracy per group (%), free decoding

| Model | Precision | Plain | Nested | Enum | Number | Optional | Unicode | Menu |
|---|---:|---:|---:|---:|---:|---:|---:|---:|
| Qwen3-4B | FP16 | 100.00 | 100.00 | 100.00 | 100.00 | 100.00 | 100.00 | 100.00 |
| Qwen3-4B | INT8 | 100.00 | 96.00 | 96.00 | 100.00 | 100.00 | 100.00 | 100.00 |
| Qwen3-4B | NF4 | 100.00 | 76.00 | 100.00 | 100.00 | 100.00 | 96.00 | 96.00 |
| Qwen3-4B | GPTQ | 100.00 | 76.00 | 100.00 | 100.00 | 100.00 | 100.00 | 96.00 |
| Qwen3-4B | AWQ | 100.00 | 76.00 | 96.00 | 100.00 | 100.00 | 96.00 | 100.00 |
| Qwen3-8B | FP16 | 100.00 | 100.00 | 100.00 | 100.00 | 100.00 | 100.00 | 100.00 |
| Qwen3-8B | INT8 | 100.00 | 100.00 | 100.00 | 100.00 | 100.00 | 100.00 | 100.00 |
| Qwen3-8B | NF4 | 100.00

## After the run: what goes into the repository

1. Download **`update12_results_A.zip`** and **`update12_results_B.zip`** from the Output tab of each saved version, and each notebook with its outputs.
2. Merge the two parts: `results/meta/`, `results/scores/` and the new rows of `results/summary.csv`; the raw answers in `results/raw/` stay out of git.
3. Analyse by group and level (Update 12), add the GPU time to `notes/compute_log.md`, any design choice to `notes/decisions.md`, and the entry to `UPDATES.md`, then commit and push.